In [1]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import importlib
import functions as f
from scintkit.preprocessing.format import temp_formating
from scintkit.services.phase_detrend import detect_sampling_rate
from pathlib import Path
import time
import CONFIG as cf

importlib.reload(f)
importlib.reload(cf)


<module 'CONFIG' from 'c:\\Users\\irees\\Downloads\\Summer_learning\\research26\\scintkit_summer\\src\\scintkit\\space_receiver_processing\\CONFIG.py'>

In [2]:

start = time.time()

print(f'started at {start- start}')
#create file organization code:

files = f.find_files(cf.input_directory)

receiverA_files, receiverB_files = f.org_receivers(files, cf.r_latitude, cf.r_longitude, cf.lat_tol, cf.lon_tol)

    #finally have all the receiver data organized into 2 seperate files 

# ----- Verification -----
print(f"Found {len(files)} total files")
print(f"Receiver A: {len(receiverA_files)} files")
print(f"Receiver B: {len(receiverB_files)} files")

if len(receiverA_files) == 0:
    raise ValueError("No files were assigned to Receiver A.")

if len(receiverB_files) == 0:
    raise ValueError("No files were assigned to Receiver B.")

print(f"time to load in files: {time.time() - start:.3f} seconds")


#create file pairing code:

paired_files = f.pair_receiver_files(receiverA_files, receiverB_files, cf)
print(f"Created {len(paired_files)} valid file pairs")


all_scint = []
for fileA, fileB in paired_files:

    print(f"\nProcessing:")
    print(fileA)
    print(fileB)

    #read single files # import pqs
    dfa = pd.read_parquet(fileA)
    dfb = pd.read_parquet(fileB)

    print(f"time to read files: {time.time() - start:.3f} seconds")


    dfa = dfa.sort_values("datetime").reset_index(drop=True)
    dfb = dfb.sort_values("datetime").reset_index(drop=True)

    # filter dfs to contain certain elevation
    dfa = dfa[dfa['elev'] > 20].copy()
    dfb = dfb[dfb['elev'] > 20].copy()

    # individual sampling rates
    dfa = temp_formating(dfa)
    dfb = temp_formating(dfb)

    samp_ra = detect_sampling_rate(dfa)
    dt = 1 / samp_ra

    #add s4 filtering here in method 2

    # merge 2 receiver dfs
    merged = dfa.merge(dfb, on=["datetime", "svid", "cons"], suffixes=("_A", "_B"))
    merged['snr_diff'] = abs(merged['snr1_A'] - merged['snr1_B'])
    print(f"time to merge dfs: {time.time() - start:.3f} seconds")

    

    thresh = cf.thresh  # threshold for s4 scintillation measurement

    #### start cross corr file creation

    rstart = time.time()

    scint = []

    sat_groups = merged.groupby(['svid', 'cons'])

    for (svid, cons), sat_group in sat_groups:

        # group the data into different satellites
        sat_group = f.datetime_to_seconds(sat_group)

        min_groups = sat_group.groupby(sat_group['datetime'].dt.floor('min'))

        # with the chosen satellite for this iteration
        # find s4 and then determine scintillation
        for min, group in min_groups:
            
            group = f.handle_nan(group, cf.nan_method)
            #add nan processing
            if len(group) < 10: #makes sure there is enough samples to actually process data
                continue

            sig_lina = f.db2lin(group['snr1_A'])
            s4a = np.std(sig_lina) / np.mean(sig_lina)

            sig_linb = f.db2lin(group['snr1_B'])
            s4b = np.std(sig_linb) / np.mean(sig_linb)

            if s4a > thresh or s4b > thresh:

                # check threshold of scintillation and compute correlation and run auto correlation
                correlation, lag_b, cor_norm, lag_norm = f.cross_correlation(group["snr1_A"], group["snr1_B"])
                autoA_max, autoA_lagb, autoA_cor, autoA_lags = f.cross_correlation(group['snr1_A'], group['snr1_A'])
                autoB_max, autoB_lagb, autoB_cor, autoB_lags = f.cross_correlation(group['snr1_B'], group['snr1_B'])

                time_delay = lag_b * dt

                scint.append({
                    'minute': min,
                    'prn' : group['prn_B'].iloc[0],
                    's4A': s4a,
                    's4B': s4b,
                    

                    #adding elev and azim
                    'elev' : group['elev_A'].mean(),
                    'azim' : group['azim_A'].mean(),
                    
                    #adding location, using the location at the start of each minute not the mean, can be changed
                    'rAloc' : (group['lat_A'].iloc[0]/10000, group['lon_A'].iloc[0]/10000, group['hei_A'].mean()/1000),
                    'rBloc' : (group['lat_B'].iloc[0]/10000, group['lon_B'].iloc[0]/10000, group['hei_B'].mean()/1000),

                    'auto_cor_A' : autoA_cor,
                    'auto_cor_Amax' : autoA_max,
                    'auto_cor_B' : autoB_cor,
                    'auto_cor_Bmax' : autoB_max,


                    'corr_norm': cor_norm,
                    'lag_norm': lag_norm,
                    'max_corr': correlation,
                    'best_lag': lag_b,
                    'time_delay': time_delay
                })

    all_scint.extend(scint)

# create dataframe storing all scintillation events with distance
cross_cor = pd.DataFrame(all_scint)

print(f"Processed {len(cross_cor)} scintillation events")
print(f"Runtime: {time.time() - rstart:.3f} seconds")


# add distance
cross_cor['distance (km)'] = cross_cor.apply(lambda row: f.calc_dist(row['rAloc'], row['rBloc']), axis = 1)

print(f"calculated distances")
print(f"Runtime: {time.time() - rstart:.3f} seconds")


#use extend to aviod appending lists


started at 0.0
Found 98 total files
Receiver A: 49 files
Receiver B: 49 files
time to load in files: 0.005 seconds
Skipping scintpi3_20250325_1552_359062.2500W_72122.5234S_v326d_lvl0.pq
Skipping scintpi3_20250326_1552_359062.2188W_72122.3750S_v326d_lvl0.pq
Skipping scintpi3_20250327_1552_359062.1562W_72122.2031S_v326d_lvl0.pq
Skipping scintpi3_20250328_1552_359062.4375W_72122.2578S_v326d_lvl0.pq
Skipping scintpi3_20250329_1552_359062.2188W_72122.2188S_v326d_lvl0.pq
Skipping scintpi3_20250330_1552_359062.3438W_72121.8984S_v326d_lvl0.pq
Skipping scintpi3_20250331_1552_359062.3125W_72122.4062S_v326d_lvl0.pq
Created 42 valid file pairs

Processing:
C:\Users\irees\Downloads\Summer_learning\research26\1_month_test-selected_lvl0_25-31\scintpi3_20250325_0000_359062.5312W_72122.7500S_v326d_lvl0.pq
C:\Users\irees\Downloads\Summer_learning\research26\1_month_test-selected_lvl0_25-31\scintpi3_20250325_0000_359072.5625W_72127.2500S_v326d_lvl0.pq
time to read files: 1.814 seconds
time to merge dfs: 

ValueError: Invalid NaN handling method: '"drop"'. Choose 'drop', 'interpolate', or 'none'.

In [ ]:
from pathlib import Path

output_folder = Path(cf.output_folder)
output_folder.mkdir(parents=True, exist_ok=True)

output_path = output_folder / cf.cross_correlation_file

cross_cor.to_parquet(output_path, index=False)

print(f"Total Time: {time.time() - start:.3f} seconds")
